# 005 — Regenerar splits con dataset completo

Este notebook reemplaza los splits generados en el notebook 00 (que usaban solo 200 muestras).

**Qué hace:**
1. Carga el dataset completo (~30k imágenes)
2. Segrega 25 imágenes de visual test, fuera de todo split
3. Con las restantes hace un split 80-15-5
4. Guarda subsets de entrenamiento (10k / 1k / 1k) para experimentos rápidos
5. Verifica integridad de todos los archivos generados

**Archivos generados:**
- `data/visual_test_indices.json` — 25 índices segregados para análisis de interpretabilidad
- `data/splits/train_indices.json` — ~24k índices (80% completo)
- `data/splits/val_indices.json` — ~4.5k índices (15% completo)
- `data/splits/test_indices.json` — ~1.5k índices (5% completo)
- `data/splits/train_sub_indices.json` — primeros 10k de train
- `data/splits/val_sub_indices.json` — primeros 1k de val
- `data/splits/test_sub_indices.json` — primeros 1k de test

**IMPORTANTE:** correr este notebook una sola vez. Una vez generados los archivos, no volver a ejecutarlo.


total imagenes:

- Disponibles: 30.608 (30.633 − 25, antes de filtrar impression vacíos)
- Train (80%): ~24.486
- Val (15%): ~4.591
- Test (5%): ~1.531

In [1]:
from pathlib import Path
import os
import sys

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CWD:", Path.cwd())

PROJECT_ROOT: /home/felisl/Escritorio/feli/vision/segunda_parte/tp_final/image-captioning
CWD: /home/felisl/Escritorio/feli/vision/segunda_parte/tp_final/image-captioning


## 1. Cargar dataset completo

In [2]:
from src.data.utils import load_mimic_dataset

ds = load_mimic_dataset(cache_dir="data/hf_cache")
train = ds["train"]

print("Dataset cargado.")
print("Total de muestras:", len(train))
print("Columnas:", train.column_names)

Dataset cargado.
Total de muestras: 30633
Columnas: ['image', 'findings', 'impression']


## 2. Obtener índices válidos (impression no vacío)

In [3]:
from src.data.utils import get_valid_indices

valid_indices = get_valid_indices(train, text_col="impression")

print("Total de muestras válidas:", len(valid_indices))
print("Muestras descartadas (impression vacío):", len(train) - len(valid_indices))

Total de muestras válidas: 30623
Muestras descartadas (impression vacío): 10


## 3. Segregar 25 imágenes de visual test

Estas imágenes quedan **fuera de todo split** y se usan exclusivamente para el análisis
de interpretabilidad (cross-attention y Grad-CAM) antes y después del fine-tuning.

In [4]:
import random
from src.data.utils import save_json

N_VISUAL_TEST = 25
SEED = 42

rng = random.Random(SEED)
shuffled = valid_indices.copy()
rng.shuffle(shuffled)

visual_test_indices = sorted(shuffled[:N_VISUAL_TEST])

save_json(visual_test_indices, "data/visual_test_indices.json")

print("Índices de visual test:", visual_test_indices)
print("Guardados en data/visual_test_indices.json")

Índices de visual test: [731, 2296, 2838, 7252, 8184, 8305, 8559, 9975, 12542, 13266, 13636, 15809, 16019, 16332, 16393, 17283, 19611, 21402, 21880, 23628, 24266, 26073, 27208, 29211, 29326]
Guardados en data/visual_test_indices.json


## 4. Generar splits 80-15-5 excluyendo los visual test

In [5]:
from src.data.split_generator import generate_splits

TRAIN_RATIO = 0.80
VAL_RATIO   = 0.15
# TEST usa el resto (0.05)

n_available = len(valid_indices) - N_VISUAL_TEST
train_size  = int(TRAIN_RATIO * n_available)
val_size    = int(VAL_RATIO   * n_available)
test_size   = n_available - train_size - val_size

print(f"Muestras disponibles (sin visual test): {n_available}")
print(f"Train ({TRAIN_RATIO*100:.0f}%): {train_size}")
print(f"Val   ({VAL_RATIO*100:.0f}%):  {val_size}")
print(f"Test  ({(1-TRAIN_RATIO-VAL_RATIO)*100:.0f}%):   {test_size}")

splits = generate_splits(
    hf_split=train,
    text_col="impression",
    train_size=train_size,
    val_size=val_size,
    test_size=test_size,
    seed=SEED,
    output_dir="data/splits",
    selected_output="data/splits/_selected_unused.json",
    auto_shrink=False,
    excluded_indices=visual_test_indices,
)

print("\nSplits generados.")
print("Train:", splits["used_train_size"])
print("Val:",   splits["used_val_size"])
print("Test:",  splits["used_test_size"])

Muestras disponibles (sin visual test): 30598
Train (80%): 24478
Val   (15%):  4589
Test  (5%):   1531

Splits generados.
Train: 24478
Val: 4589
Test: 1531


## 5. Guardar subsets para experimentos rápidos

Los subsets son simplemente los primeros N índices de cada split completo.
Para aumentar el tamaño de entrenamiento basta con cambiar `TRAIN_SUB_SIZE`.

In [6]:
TRAIN_SUB_SIZE = 15000
VAL_SUB_SIZE   = 1000
TEST_SUB_SIZE  = 1000

train_sub = splits["train"][:TRAIN_SUB_SIZE]
val_sub   = splits["val"][:VAL_SUB_SIZE]
test_sub  = splits["test"][:TEST_SUB_SIZE]

save_json(train_sub, "data/splits/train_sub_indices.json")
save_json(val_sub,   "data/splits/val_sub_indices.json")
save_json(test_sub,  "data/splits/test_sub_indices.json")

print(f"Subsets guardados:")
print(f"  train_sub: {len(train_sub):>6}  ({len(train_sub)/len(splits['train'])*100:.1f}% del train completo)")
print(f"  val_sub:   {len(val_sub):>6}")
print(f"  test_sub:  {len(test_sub):>6}")

Subsets guardados:
  train_sub:  15000  (61.3% del train completo)
  val_sub:     1000
  test_sub:    1000


## 6. Verificación de integridad

Si alguna verificación falla, la celda lanza una excepción y detiene la ejecución.

In [7]:
from src.data.utils import load_json

vt  = set(load_json("data/visual_test_indices.json"))
tr  = set(load_json("data/splits/train_indices.json"))
val = set(load_json("data/splits/val_indices.json"))
te  = set(load_json("data/splits/test_indices.json"))
trs = set(load_json("data/splits/train_sub_indices.json"))
vas = set(load_json("data/splits/val_sub_indices.json"))
tes = set(load_json("data/splits/test_sub_indices.json"))

n_total = len(train)

checks = [
    ("visual_test sin solapamiento con train",   vt.isdisjoint(tr)),
    ("visual_test sin solapamiento con val",     vt.isdisjoint(val)),
    ("visual_test sin solapamiento con test",    vt.isdisjoint(te)),
    ("train sin solapamiento con val",           tr.isdisjoint(val)),
    ("train sin solapamiento con test",          tr.isdisjoint(te)),
    ("val sin solapamiento con test",            val.isdisjoint(te)),
    ("train_sub es subconjunto de train",        trs.issubset(tr)),
    ("val_sub es subconjunto de val",            vas.issubset(val)),
    ("test_sub es subconjunto de test",          tes.issubset(te)),
    ("visual_test tiene 25 indices",             len(vt) == N_VISUAL_TEST),
    ("train_sub tiene tamaño correcto",          len(trs) == TRAIN_SUB_SIZE),
    ("val_sub tiene tamaño correcto",            len(vas) == VAL_SUB_SIZE),
    ("test_sub tiene tamaño correcto",           len(tes) == TEST_SUB_SIZE),
    ("todos los indices dentro de rango",
     all(0 <= i < n_total for i in vt | tr | val | te)),
]

failed = [(name, ok) for name, ok in checks if not ok]

for name, ok in checks:
    status = "OK" if ok else "FALLO"
    print(f"[{status}] {name}")

if failed:
    raise RuntimeError(
        f"Fallaron {len(failed)} verificaciones: {[n for n, _ in failed]}"
    )

print("\nTodas las verificaciones pasaron correctamente.")

[OK] visual_test sin solapamiento con train
[OK] visual_test sin solapamiento con val
[OK] visual_test sin solapamiento con test
[OK] train sin solapamiento con val
[OK] train sin solapamiento con test
[OK] val sin solapamiento con test
[OK] train_sub es subconjunto de train
[OK] val_sub es subconjunto de val
[OK] test_sub es subconjunto de test
[OK] visual_test tiene 25 indices
[OK] train_sub tiene tamaño correcto
[OK] val_sub tiene tamaño correcto
[OK] test_sub tiene tamaño correcto
[OK] todos los indices dentro de rango

Todas las verificaciones pasaron correctamente.


## 7. Verificar dataloaders

In [8]:
from transformers import BlipProcessor
from src.data.dataloader import create_dataloader

processor = BlipProcessor.from_pretrained("models/blip_base")

train_loader = create_dataloader(
    hf_split=train,
    indices_path="data/splits/train_sub_indices.json",
    processor=processor,
    text_col="impression",
    batch_size=4,
    shuffle=True,
)

val_loader = create_dataloader(
    hf_split=train,
    indices_path="data/splits/val_sub_indices.json",
    processor=processor,
    text_col="impression",
    batch_size=4,
    shuffle=False,
)

print("Train loader — batches:", len(train_loader))
print("Val loader   — batches:", len(val_loader))

Train loader — batches: 3750
Val loader   — batches: 250


In [9]:
batch = next(iter(train_loader))

expected_keys = {"input_ids", "attention_mask", "pixel_values", "labels"}
missing_keys = expected_keys - set(batch.keys())

if missing_keys:
    raise RuntimeError(f"Faltan claves en el batch: {missing_keys}")

print("Claves del batch:", list(batch.keys()))
for key, value in batch.items():
    if hasattr(value, "shape"):
        print(f"  {key}: {value.shape}")

assert batch["pixel_values"].shape[-2:] == (384, 384), \
    f"Shape inesperado de pixel_values: {batch['pixel_values'].shape}"

print("\nBatch verificado correctamente.")

Claves del batch: ['input_ids', 'attention_mask', 'pixel_values', 'labels', 'idx', 'text']
  input_ids: torch.Size([4, 128])
  attention_mask: torch.Size([4, 128])
  pixel_values: torch.Size([4, 3, 384, 384])
  labels: torch.Size([4, 128])
  idx: torch.Size([4])

Batch verificado correctamente.


## 8. Resumen final

In [10]:
print("=" * 60)
print("Splits generados correctamente")
print("=" * 60)
print(f"Dataset total:         {len(train):>8}")
print(f"Visual test (segregados): {len(vt):>5}  → data/visual_test_indices.json")
print(f"Train completo:        {len(tr):>8}  → data/splits/train_indices.json")
print(f"Val completo:          {len(val):>8}  → data/splits/val_indices.json")
print(f"Test completo:         {len(te):>8}  → data/splits/test_indices.json")
print(f"Train sub ({TRAIN_SUB_SIZE}):      {len(trs):>8}  → data/splits/train_sub_indices.json")
print(f"Val sub ({VAL_SUB_SIZE}):        {len(vas):>8}  → data/splits/val_sub_indices.json")
print(f"Test sub ({TEST_SUB_SIZE}):       {len(tes):>8}  → data/splits/test_sub_indices.json")
print("=" * 60)
print("Próximo paso: notebook 02_baseline_radiografias.ipynb")
print("Usar visual_test_indices.json como imágenes de análisis.")

Splits generados correctamente
Dataset total:            30633
Visual test (segregados):    25  → data/visual_test_indices.json
Train completo:           24478  → data/splits/train_indices.json
Val completo:              4589  → data/splits/val_indices.json
Test completo:             1531  → data/splits/test_indices.json
Train sub (15000):         15000  → data/splits/train_sub_indices.json
Val sub (1000):            1000  → data/splits/val_sub_indices.json
Test sub (1000):           1000  → data/splits/test_sub_indices.json
Próximo paso: notebook 02_baseline_radiografias.ipynb
Usar visual_test_indices.json como imágenes de análisis.
